# 🌾 Atelier IA Agricole — 03. TinyLLM & TinyVLM: les modèles génératifs les plus petits

Jusqu'ici, on a utilisé des modèles **milliard** de paramètres (SLM et LLM). Ce notebook explore l'**autre extrême**: les
plus petits modèles génératifs disponibles sur Hugging Face — un **TinyLLM** (texte) et un
**TinyVLM** (image + texte), chacun sous les **300 millions** de paramètres.

**Objectif:** réutiliser **les mêmes tâches et le même jeu de données** que les notebooks 01
et 02, mais avec des modèles bien plus petits, pour **observer concrètement** la perte de
qualité — et l'effet encore plus marqué de réglages comme la température ou le few-shot sur
un modèle minuscule.


In [18]:
# === Configuration de l'environnement (exécuter en premier) ===
import os, sys, subprocess

# Sommes-nous dans Google Colab?
try:
    import google.colab
    IS_COLAB = True
except Exception:
    IS_COLAB = False

# MODE_DEMO = True  -> utilise de tout petits modèles / jeux de données réduits
# (utile pour tester rapidement, ou hors GPU). Mettez False pour l'atelier complet.
# La variable d'environnement ATELIER_DEMO permet de forcer le mode pour les tests.
MODE_DEMO = os.environ.get("ATELIER_DEMO", "0") == "1"

def pip_install(*paquets):
    """Installe des paquets pip de façon silencieuse (Colab ou local)."""
    cmd = [sys.executable, "-m", "pip", "install", "-q", *paquets]
    print("Installation:", " ".join(paquets), "...")
    subprocess.run(cmd, check=False)

print(f"IS_COLAB = {IS_COLAB}")
print(f"MODE_DEMO = {MODE_DEMO}")
print("Python:", sys.version.split()[0])

IS_COLAB = True
MODE_DEMO = False
Python: 3.12.13


In [19]:
pip_install("transformers>=4.56", "accelerate", "pillow")
import gc
from transformers import pipeline
print("✅ Bibliothèques prêtes.")

Installation: transformers>=4.56 accelerate pillow ...
✅ Bibliothèques prêtes.


In [20]:
pip_install("kagglehub", "pandas")
import kagglehub

def telecharger_dataset_kaggle(reference):
    """Télécharge un jeu de données Kaggle public et renvoie son dossier local.
    Renvoie None si Kaggle est injoignable (un échantillon de secours prend alors le relais)."""
    try:
        dossier = kagglehub.dataset_download(reference)
        print(f"✅ Jeu de données Kaggle prêt: {reference}")
        return dossier
    except Exception as e:
        print(f"⚠️ Kaggle indisponible ({e}) → utilisation d'un échantillon de secours.")
        return None

Installation: kagglehub pandas ...


In [21]:
pip_install("pillow")
import os, random, io, urllib.request
from PIL import Image

REFERENCE_IMAGES_KAGGLE = "rashikrahmanpritom/plant-disease-recognition-dataset"
URLS_SECOURS = [
    "https://upload.wikimedia.org/wikipedia/commons/thumb/1/16/Leaf_curl_on_peach.jpg/960px-Leaf_curl_on_peach.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/f/f6/Jowar_leaf_infested_by_pest_or_disease.jpg/960px-Jowar_leaf_infested_by_pest_or_disease.jpg",
]

def echantillon_images_plantes(n=10):
    """Renvoie une liste de (image PIL, label): n photos du jeu Kaggle Healthy/Powdery/Rust;
    à défaut, quelques URLs publiques; et en dernier recours une image synthétique hors-ligne.
    Ainsi la liste renvoyée n'est jamais vide, même sans réseau."""
    dossier = telecharger_dataset_kaggle(REFERENCE_IMAGES_KAGGLE)
    if dossier:
        try:
            base = os.path.join(dossier, "Test", "Test")
            fichiers = []
            for classe in sorted(os.listdir(base)):
                dossier_c = os.path.join(base, classe)
                if not os.path.isdir(dossier_c):
                    continue
                for nom in sorted(os.listdir(dossier_c))[:20]:
                    fichiers.append((os.path.join(dossier_c, nom), classe))
            random.Random(42).shuffle(fichiers)
            images = [(Image.open(chemin).convert("RGB"), label) for chemin, label in fichiers[:n]]
            if images:
                return images
            print("⚠️ Jeu Kaggle vide/inattendu → images de secours.")
        except Exception as e:
            print(f"⚠️ Lecture du jeu Kaggle impossible ({e}) → images de secours.")

    entetes = {"User-Agent": "Mozilla/5.0 (AtelierIA-Agricole; educatif)"}
    images = []
    for url in URLS_SECOURS[:n]:
        try:
            req = urllib.request.Request(url, headers=entetes)
            with urllib.request.urlopen(req, timeout=20) as r:
                images.append((Image.open(io.BytesIO(r.read())).convert("RGB"), "inconnu"))
        except Exception as e:
            print("⚠️ Image ignorée:", e)

    # Dernier recours 100% hors-ligne: si Kaggle ET les URLs échouent, on fabrique au moins
    # une image synthétique pour que le notebook aille jusqu'au bout (pas d'IndexError sur
    # echantillon[0], liste jamais vide).
    if not images:
        from PIL import ImageDraw
        print("⚠️ Aucune image en ligne → image de secours générée localement.")
        for _ in range(max(1, min(n, 3))):
            img = Image.new("RGB", (256, 256), (150, 170, 90))
            ImageDraw.Draw(img).ellipse([40, 40, 216, 216], fill=(70, 110, 50))
            images.append((img, "inconnu"))
    return images


Installation: pillow ...


## 1. TinyLLM: `SmolLM2-135M-Instruct`

135 millions de paramètres — environ **8 fois plus petit** que le SLM du notebook 01.


In [22]:
MODELE_TINYLLM = "HuggingFaceTB/SmolLM2-135M-Instruct"
pipe_tinyllm = pipeline("text-generation", model=MODELE_TINYLLM, dtype="auto")
n_params = pipe_tinyllm.model.num_parameters() / 1e6
print(f"✅ {MODELE_TINYLLM} chargé (~{n_params:.0f} M paramètres)")

def chat_tiny(prompt, temperature=0.0, max_new_tokens=60):
    messages = [{"role": "user", "content": prompt}]
    options = {"max_new_tokens": max_new_tokens}
    options.update({"do_sample": True, "temperature": temperature} if temperature > 0
                    else {"do_sample": False, "temperature": 1.0, "top_p": 1.0, "top_k": 50})
    sortie = pipe_tinyllm(messages, **options)
    return sortie[0]["generated_text"][-1]["content"].strip()

print(chat_tiny("Qu'est-ce que la rotation des cultures?"))

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ HuggingFaceTB/SmolLM2-135M-Instruct chargé (~135 M paramètres)
La rotation des cultures, also known as the rotation of cultures, is a cultural phenomenon that occurs when a group of people from different countries or regions come together to share their customs, traditions, and ways of life. It is a way of thinking and a way of life that is unique to each individual


## 2. TinyLLM sur le jeu de données Kaggle (crop recommendation)

On reprend **exactement** la tâche du notebook 01 (recommander une culture à partir de
mesures de sol), en **zero-shot**, avec le TinyLLM.


In [23]:
import pandas as pd

ECHANTILLON_SECOURS = [
    {"N": 20, "P": 129, "K": 201, "temperature": 23.4, "humidity": 91.7, "ph": 5.59, "rainfall": 116.1, "label": "apple"},
    {"N": 100, "P": 80, "K": 52, "temperature": 27.5, "humidity": 77.3, "ph": 6.05, "rainfall": 110.3, "label": "banana"},
    {"N": 43, "P": 68, "K": 20, "temperature": 29.6, "humidity": 66.2, "ph": 7.5, "rainfall": 69.4, "label": "blackgram"},
    {"N": 43, "P": 68, "K": 81, "temperature": 17.5, "humidity": 17.9, "ph": 6.76, "rainfall": 78.9, "label": "chickpea"},
    {"N": 21, "P": 20, "K": 31, "temperature": 25.6, "humidity": 99.7, "ph": 5.86, "rainfall": 165.8, "label": "coconut"},
    {"N": 107, "P": 31, "K": 31, "temperature": 23.2, "humidity": 53.0, "ph": 6.77, "rainfall": 153.1, "label": "coffee"},
    {"N": 122, "P": 40, "K": 17, "temperature": 25.0, "humidity": 81.3, "ph": 6.85, "rainfall": 80.0, "label": "cotton"},
    {"N": 22, "P": 133, "K": 201, "temperature": 23.8, "humidity": 80.1, "ph": 6.0, "rainfall": 67.3, "label": "grapes"},
    {"N": 80, "P": 43, "K": 43, "temperature": 23.8, "humidity": 74.4, "ph": 6.01, "rainfall": 172.6, "label": "jute"},
    {"N": 24, "P": 67, "K": 22, "temperature": 20.1, "humidity": 22.9, "ph": 5.62, "rainfall": 104.6, "label": "kidneybeans"},
    {"N": 18, "P": 66, "K": 22, "temperature": 25.9, "humidity": 67.6, "ph": 6.35, "rainfall": 47.9, "label": "lentil"},
    {"N": 76, "P": 48, "K": 18, "temperature": 19.3, "humidity": 69.6, "ph": 5.78, "rainfall": 83.2, "label": "maize"},
    {"N": 18, "P": 26, "K": 31, "temperature": 32.6, "humidity": 47.7, "ph": 5.42, "rainfall": 91.1, "label": "mango"},
    {"N": 25, "P": 51, "K": 18, "temperature": 27.8, "humidity": 54.8, "ph": 9.46, "rainfall": 50.3, "label": "mothbeans"},
    {"N": 21, "P": 44, "K": 18, "temperature": 27.1, "humidity": 86.9, "ph": 7.13, "rainfall": 50.5, "label": "mungbean"},
    {"N": 100, "P": 17, "K": 48, "temperature": 29.7, "humidity": 94.3, "ph": 6.37, "rainfall": 26.5, "label": "muskmelon"},
    {"N": 12, "P": 20, "K": 10, "temperature": 24.5, "humidity": 93.1, "ph": 6.53, "rainfall": 109.5, "label": "orange"},
    {"N": 54, "P": 67, "K": 52, "temperature": 35.7, "humidity": 93.3, "ph": 6.59, "rainfall": 141.3, "label": "papaya"},
    {"N": 27, "P": 71, "K": 23, "temperature": 23.5, "humidity": 46.5, "ph": 7.11, "rainfall": 150.9, "label": "pigeonpeas"},
    {"N": 21, "P": 21, "K": 38, "temperature": 22.6, "humidity": 89.3, "ph": 6.33, "rainfall": 104.9, "label": "pomegranate"},
    {"N": 81, "P": 53, "K": 42, "temperature": 23.7, "humidity": 81.0, "ph": 5.18, "rainfall": 233.7, "label": "rice"},
    {"N": 103, "P": 16, "K": 49, "temperature": 24.1, "humidity": 81.6, "ph": 6.92, "rainfall": 51.8, "label": "watermelon"},
]

dossier = telecharger_dataset_kaggle("atharvaingle/crop-recommendation-dataset")
df = None
if dossier:
    try:
        df = pd.read_csv(f"{dossier}/Crop_recommendation.csv")
    except Exception as e:
        print(f"⚠️ Lecture du CSV Kaggle impossible ({e}) → échantillon de secours.")
if df is None:
    df = pd.DataFrame(ECHANTILLON_SECOURS)

def ligne_en_texte(ligne):
    return (f"N={ligne['N']}, P={ligne['P']}, K={ligne['K']}, "
            f"température={ligne['temperature']:.1f}°C, humidité={ligne['humidity']:.0f}%, "
            f"pH={ligne['ph']:.1f}, pluie={ligne['rainfall']:.0f}mm")

# Donner la liste FERMÉE des cultures possibles aide beaucoup un petit modèle: sans elle,
# il invente souvent des mots hors sujet plutôt que de choisir une vraie culture du jeu.
LISTE_CULTURES = sorted(df["label"].unique())
CULTURES_TXT = ", ".join(LISTE_CULTURES)

N_EXEMPLES = 3 if MODE_DEMO else 10
echantillon_crop = df.sample(n=N_EXEMPLES, random_state=42)

for _, ligne in echantillon_crop.iterrows():
    prompt = (f"Mesures: {ligne_en_texte(ligne)}. Quelle culture recommandes-tu, en "
              f"choisissant UNIQUEMENT dans cette liste: {CULTURES_TXT}?\n"
              "Réponds uniquement juste le nom de la culture.")
    prediction = chat_tiny(prompt, max_new_tokens=20)
    print(f"Vrai: {ligne['label']:12s} | TinyLLM: {prediction}")

Using Colab cache for faster access to the 'crop-recommendation-dataset' dataset.


[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Jeu de données Kaggle prêt: atharvaingle/crop-recommendation-dataset


[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Vrai: muskmelon    | TinyLLM: "Mesures: N=101, P=17, K=47,


...

Vrai: apple        | TinyLLM: "Mesures: N=35, P=145, K=195


[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Vrai: mothbeans    | TinyLLM: "Mesures: N=22, P=44, K=24, temp


[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Vrai: mungbean     | TinyLLM: Les cultures humaines, nous avons un nombre de cultures humaines qui s
Vrai: lentil       | TinyLLM: Les cultures humaines, nous avons un nombre de cultures humaines qui s


> 💡 Comparez ces réponses à celles du notebook 01 (SLM ~1 Md) sur la même tâche: le TinyLLM
> se trompe ou dérape plus souvent — c'est le compromis taille/qualité en action.


In [24]:
del pipe_tinyllm
gc.collect()

30

## 3. TinyVLM: `SmolVLM-256M-Instruct`

256 millions de paramètres — environ **2 fois plus petit** que le VLM du notebook 02.


In [25]:
MODELE_TINYVLM = "HuggingFaceTB/SmolVLM-256M-Instruct"
pipe_tinyvlm = pipeline("image-text-to-text", model=MODELE_TINYVLM, dtype="auto")
pipe_tinyvlm.image_processor.do_image_splitting = False
n_params = pipe_tinyvlm.model.num_parameters() / 1e6
print(f"✅ {MODELE_TINYVLM} chargé (~{n_params:.0f} M paramètres)")

def demander_image_tiny(image, question, max_new_tokens=40):
    messages = [{"role": "user", "content": [{"type": "image", "image": image},
                                              {"type": "text", "text": question}]}]
    sortie = pipe_tinyvlm(text=messages, max_new_tokens=max_new_tokens)
    return sortie[0]["generated_text"][-1]["content"].strip()

Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

✅ HuggingFaceTB/SmolVLM-256M-Instruct chargé (~256 M paramètres)


## 4. Fine-tuning: spécialiser le TinyLLM sur nos cultures

En section 2, le TinyLLM « brut » se trompe presque toujours: il n'a jamais appris le lien
entre mesures de sol et culture. Le **fine-tuning** ré-entraîne le modèle sur NOS exemples
pour l'adapter à la tâche.

135 M de paramètres, ça reste petit: pour une démonstration **nette et rapide**, on cible
**quelques cultures bien distinctes** (avec les 22 d'un coup, il lui faudrait beaucoup plus de
données et de calcul). On utilise **LoRA** — au lieu de ré-entraîner les 135 M de paramètres,
on n'ajoute que quelques petites matrices: rapide, léger, parfait pour un GPU Colab gratuit.

**Plan:** mesurer la précision AVANT, entraîner quelques minutes, puis re-mesurer sur le
**même** jeu de test. On veut voir la précision **grimper**.


In [26]:
# Le fine-tuning entraîne les poids: on repart du modèle « brut » (pas d'un pipeline).
# On libère d'abord les modèles des sections précédentes pour la RAM.
for _v in ("pipe_tinyvlm", "pipe_tinyllm"):
    if _v in globals():
        del globals()[_v]
gc.collect()

pip_install("peft")
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

GPU_DISPO = torch.cuda.is_available()
DEVICE = "cuda" if GPU_DISPO else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODELE_TINYLLM)

# Quelques cultures aux profils sol/climat bien différents (démonstration nette).
CULTURES_FT = ["rice", "apple", "cotton", "grapes", "maize", "banana"]
N_CULTURES = 4 if MODE_DEMO else 6
CULTURES_FT = [c for c in CULTURES_FT if c in set(df["label"])][:N_CULTURES]

# Prompt COMPACT (mesures -> culture). La réponse attendue est le nom de la culture.
def prompt_ft(l):
    return (f"Sol: N={l['N']}, P={l['P']}, K={l['K']}, temp={l['temperature']:.0f}C, "
            f"humidite={l['humidity']:.0f}%, pH={l['ph']:.1f}, pluie={l['rainfall']:.0f}mm. Culture?")

def messages_ft(l, avec_reponse=True):
    msgs = [{"role": "user", "content": prompt_ft(l)}]
    if avec_reponse:
        msgs.append({"role": "assistant", "content": str(l["label"])})
    return msgs

# Jeu ÉQUILIBRÉ: K_TEST exemples/culture pour le test, K_TRAIN pour l'entrainement.
K_TEST = 4
K_TRAIN = 30 if MODE_DEMO else 80
tr, te = [], []
for c in CULTURES_FT:
    sub = df[df["label"] == c].sample(frac=1.0, random_state=0)
    te.append(sub.iloc[:K_TEST])
    tr.append(sub.iloc[K_TEST:K_TEST + K_TRAIN])
df_test = pd.concat(te)
df_train = pd.concat(tr).sample(frac=1.0, random_state=1)

# Le fine-tuning a besoin du vrai jeu Kaggle. Sans lui (petit échantillon de secours), on saute.
ASSEZ = len(df_train) >= 2 * len(CULTURES_FT) and len(df_test) >= len(CULTURES_FT)
if ASSEZ:
    print(f"Cultures ciblées: {CULTURES_FT}")
    print(f"{len(df_train)} exemples d'entrainement, {len(df_test)} de test.")
else:
    print("⚠️ Pas assez de données (échantillon de secours Kaggle) pour le fine-tuning.")
    print("   Relancez cette section avec le jeu Kaggle disponible.")

@torch.no_grad()
def predire(model, l, max_new_tokens=8):
    enc = tokenizer.apply_chat_template(messages_ft(l, avec_reponse=False),
                                        add_generation_prompt=True, return_tensors="pt",
                                        return_dict=True).to(DEVICE)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()

def precision(model):
    bons = sum(str(l["label"]).lower() in predire(model, l) for _, l in df_test.iterrows())
    return bons / len(df_test)

Installation: peft ...
Cultures ciblées: ['rice', 'apple', 'cotton', 'grapes', 'maize', 'banana']
480 exemples d'entrainement, 24 de test.


### Précision AVANT le fine-tuning

Le modèle brut, sur ce prompt compact, répond surtout à côté.


In [27]:
if ASSEZ:
    modele = AutoModelForCausalLM.from_pretrained(MODELE_TINYLLM, dtype=torch.float32).to(DEVICE)
    modele.eval()
    prec_avant = precision(modele)
    ex = df_test.iloc[0]
    print(f"Exemple -> vrai: {ex['label']} | le modele brut repond: '{predire(modele, ex)}'")
    print(f"🔎 Précision AVANT fine-tuning: {prec_avant:.0%}")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Exemple -> vrai: rice | le modele brut repond: 'the given data indicates that the soil is'
🔎 Précision AVANT fine-tuning: 0%


### Entraînement LoRA (on n'apprend que la réponse)

Astuce clé: on ne calcule la « perte » que sur le **nom de la culture** — la partie question
est **masquée**. Ainsi le modèle apprend à *répondre*, pas à répéter la question. Sur GPU
(Colab) l'entrainement prend quelques minutes; sur CPU c'est plus lent (mode démo réduit).


In [28]:
if ASSEZ:
    # Ensure torchao is updated to a compatible version
    pip_install("torchao>=0.16.0")

    # Masquage du prompt: labels = -100 sur la question, vrai token sur la réponse.
    def encoder(l):
        full = tokenizer(tokenizer.apply_chat_template(messages_ft(l), tokenize=False),
                         add_special_tokens=False)["input_ids"]
        pre = tokenizer(tokenizer.apply_chat_template(messages_ft(l, avec_reponse=False),
                        add_generation_prompt=True, tokenize=False),
                        add_special_tokens=False)["input_ids"]
        n = 0
        while n < len(pre) and n < len(full) and full[n] == pre[n]:
            n += 1
        return {"input_ids": full, "attention_mask": [1] * len(full),
                "labels": [-100] * n + full[n:]}

    def collate(feats):
        m = max(len(f["input_ids"]) for f in feats)
        pad = tokenizer.pad_token_id
        pack = lambda cle, rembourrage: torch.tensor(
            [f[cle] + [rembourrage] * (m - len(f[cle])) for f in feats])
        return {"input_ids": pack("input_ids", pad),
                "attention_mask": pack("attention_mask", 0),
                "labels": pack("labels", -100)}

    exemples = [encoder(l) for _, l in df_train.iterrows()]
    modele = get_peft_model(modele, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules="all-linear", task_type="CAUSAL_LM"))
    modele.print_trainable_parameters()

    args = TrainingArguments(
        output_dir="/tmp/ft_tinyllm",
        per_device_train_batch_size=4,
        num_train_epochs=4,
        learning_rate=2e-4,
        logging_steps=20,
        save_strategy="no",
        report_to="none",
        disable_tqdm=True,
    )
    Trainer(model=modele, args=args, train_dataset=exemples, data_collator=collate).train()
    print("✅ Entraînement terminé.")

Installation: torchao>=0.16.0 ...
trainable params: 4,884,480 || all params: 139,399,488 || trainable%: 3.5039
{'loss': '2.445', 'grad_norm': '2.319', 'learning_rate': '0.0001921', 'epoch': '0.1667'}
{'loss': '0.6092', 'grad_norm': '8.777', 'learning_rate': '0.0001837', 'epoch': '0.3333'}
{'loss': '0.619', 'grad_norm': '2.342', 'learning_rate': '0.0001754', 'epoch': '0.5'}
{'loss': '0.5373', 'grad_norm': '4.376', 'learning_rate': '0.0001671', 'epoch': '0.6667'}
{'loss': '0.4729', 'grad_norm': '2.297', 'learning_rate': '0.0001588', 'epoch': '0.8333'}
{'loss': '0.3991', 'grad_norm': '4.129', 'learning_rate': '0.0001504', 'epoch': '1'}
{'loss': '0.3325', 'grad_norm': '5.24', 'learning_rate': '0.0001421', 'epoch': '1.167'}
{'loss': '0.2504', 'grad_norm': '3.532', 'learning_rate': '0.0001337', 'epoch': '1.333'}
{'loss': '0.2163', 'grad_norm': '3.019', 'learning_rate': '0.0001254', 'epoch': '1.5'}
{'loss': '0.1121', 'grad_norm': '11.38', 'learning_rate': '0.0001171', 'epoch': '1.667'}
{'loss

### Précision APRÈS le fine-tuning


In [29]:
if ASSEZ:
    modele.eval()
    prec_apres = precision(modele)
    print(f"AVANT: {prec_avant:.0%}   →   APRÈS: {prec_apres:.0%}")
    print()
    for _, l in df_test.iterrows():
        marque = "✅" if str(l["label"]).lower() in predire(modele, l) else "  "
        print(f"{marque} Vrai: {l['label']:10s} | Prédit: {predire(modele, l)}")


AVANT: 0%   →   APRÈS: 100%

✅ Vrai: rice       | Prédit: rice
✅ Vrai: rice       | Prédit: rice
✅ Vrai: rice       | Prédit: rice
✅ Vrai: rice       | Prédit: rice
✅ Vrai: apple      | Prédit: apple
✅ Vrai: apple      | Prédit: apple
✅ Vrai: apple      | Prédit: apple
✅ Vrai: apple      | Prédit: apple
✅ Vrai: cotton     | Prédit: cotton
✅ Vrai: cotton     | Prédit: cotton
✅ Vrai: cotton     | Prédit: cotton
✅ Vrai: cotton     | Prédit: cotton
✅ Vrai: grapes     | Prédit: grapes
✅ Vrai: grapes     | Prédit: grapes
✅ Vrai: grapes     | Prédit: grapes
✅ Vrai: grapes     | Prédit: grapes
✅ Vrai: maize      | Prédit: maize
✅ Vrai: maize      | Prédit: maize
✅ Vrai: maize      | Prédit: maize
✅ Vrai: maize      | Prédit: maize
✅ Vrai: banana     | Prédit: banana
✅ Vrai: banana     | Prédit: banana
✅ Vrai: banana     | Prédit: banana
✅ Vrai: banana     | Prédit: banana


> 💡 En quelques minutes de **fine-tuning LoRA**, un modèle minuscule (135 M) passe de
> « incapable » à « plutôt bon » sur NOTRE tâche: la précision **grimpe nettement**. C'est ainsi
> qu'on obtient de petits assistants « métier », spécialisés, embarquables et rapides.
> (Pour les 22 cultures d'un coup, il faudrait bien plus de données et de calcul — le
> fine-tuning réduit le besoin, il ne l'annule pas.)


## ✅ Récapitulatif

- **TinyLLM** et **TinyVLM** (≤ 300M paramètres) sont les plus petits modèles génératifs
  disponibles — encore plus légers qu'un SLM/VLM (~1 Md).
- Sur la **même tâche**, ils se trompent plus souvent: la taille a un vrai coût en qualité.
- Les réglages (température, few-shot) ont un effet **encore plus marqué** sur un modèle
  minuscule que sur un modèle de taille moyenne.
- Le **fine-tuning (LoRA)** spécialise un petit modèle sur une tâche précise et **améliore
  nettement** ses prédictions, pour un coût de calcul modeste.

**➡️ Notebook suivant: `04_VLM.ipynb`** — détection ouverte, segmentation
et tracking.
